In [25]:
using LowLevelFEM

In [26]:
openGeometry("truss.geo")
#openPreProcessor()

In [27]:
mat = Material("truss")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u)

Problem("truss", :VectorField, 2, 2, Material[Material("truss", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 4, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing))

In [28]:
t = tangentVector(U, "truss")

tx, ty, tz = t[1], t[2], t[3]

T = [tx*tx tx*ty tx*ty ty*ty]

B = T ⋅ Grad(U)

E = mat.E
A = mat.A

K = ∫(B' ⋅ (E * A) ⋅ B, Γ="truss")
K[:,:]

8×8 SparseArrays.SparseMatrixCSC{Float64, Int64} with 40 stored entries:
  2500.0      866.025  -2000.0    …   -866.025       ⋅          ⋅ 
   866.025   1500.0         ⋅        -1500.0         ⋅          ⋅ 
 -2000.0         ⋅      3000.0         866.025   -500.0     -866.025
      ⋅          ⋅          ⋅        -1500.0     -866.025  -1500.0
  -500.0     -866.025   -500.0            ⋅     -2000.0         ⋅ 
  -866.025  -1500.0      866.025  …   3000.0         ⋅          ⋅ 
      ⋅          ⋅      -500.0            ⋅      2500.0      866.025
      ⋅          ⋅      -866.025          ⋅       866.025   1500.0

In [29]:
f = ∫(U ⋅ [0,-1], Γ="D")
DoFs(f)

8×1 Matrix{Float64}:
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
 -1.0

In [30]:
bcA = BoundaryCondition("A", ux=0, uy=0)
bcB = BoundaryCondition("B", uy=0);

In [31]:
u = solveField(K, f, support=[bcA, bcB]);

In [32]:
showDoFResults(f, name="f")
showDoFResults(u, name="u")

1

In [33]:
u = expandTo3D(u)

g = ∇(u)

ε = t ⋅ (g * t)

N = E * A * ε

showElementResults(N, name="N")
#plotOnBeam("truss", N, name="N graph")

2

In [34]:
openPostProcessor()